DH演習2026 · 教材 第49版 · 2026-r49 · 2026-11-12

<details><summary>教材の版と相談用情報</summary>

相談時は次の情報を伝えてください。

```text
DH演習2026 · 教材 第49版
教材ID: 2026-r49
教材日付: 2026-11-12
活動: tei_annotation
環境: Colab / Jupyter
ソース: ed8d15220a966ba64a9f2277b026569aa878bf8e
```

</details>

[同じ版の教材・変更点](https://nlp.lang.osaka-u.ac.jp/dh/practicum/editions.html)


# 第5回：人物への言及を判断し、TEIのタグで表す

上から順にセルを実行してください（Shift+Enter）。値を変えたら、そのセルから下を再実行します。

芥川龍之介の戯曲『二人小町』では、二人の女性と<ruby>黄泉<rp>（</rp><rt>よみ</rt><rp>）</rp></ruby>（死者の世界）の使が、生と死を巡って交渉します。若いときは死を免れようとし、老いて貧しくなると死を願う逆転を、暗い笑いを伴う劇として読めます。

| 登場人物 | 作品での役割 | 第1場の抜粋 |
|---|---|---|
| <ruby>小野<rp>（</rp><rt>おの</rt><rp>）</rp></ruby>の<ruby>小町<rp>（</rp><rt>こまち</rt><rp>）</rp></ruby> | 死を免れようと、玉造を身代わりに勧める女性 | 発言する |
| <ruby>玉造<rp>（</rp><rt>たまつくり</rt><rp>）</rp></ruby>の<ruby>小町<rp>（</rp><rt>こまち</rt><rp>）</rp></ruby> | 小野とは別人で、第2場では使と交渉する女性 | 話題になるが、その場にはいない |
| <ruby>黄泉<rp>（</rp><rt>よみ</rt><rp>）</rp></ruby>の<ruby>使<rp>（</rp><rt>つかい</rt><rp>）</rp></ruby> | 二人を黄泉へ迎えに来る使者 | 小野と話す |
| <ruby>神将<rp>（</rp><rt>しんしょう</rt><rp>）</rp></ruby> | 小町を守護し、使の要求を退ける存在 | 第3場に登場 |

資料：芥川龍之介『二人小町』（[TEI-EAJ aozora_tei](https://github.com/TEI-EAJ/aozora_tei/blob/77a675fc2771936f9544505d922d4cd45075338c/data/complete/tei_lib_lv3/86_tei.xml)、CC0）

## 1. 本文を読む

ノートブックは戯曲全体のTEIを取得し、第1場の発言14・15・19・20を表示します。このTEIの本文には、人物を指す表現のタグがありません。

作業用に二つの変更を加えます。ルビの読みを除き、人物一覧（`listPerson`）のIDを`ono`・`tama`・`messenger`・`guards`に付け替えます。

In [ ]:
#@title 操作：本文を取得して表示する
import requests
from lxml import etree

url = "https://raw.githubusercontent.com/TEI-EAJ/aozora_tei/77a675fc2771936f9544505d922d4cd45075338c/data/complete/tei_lib_lv3/86_tei.xml"
T = "{http://www.tei-c.org/ns/1.0}"
XML_ID = "{http://www.w3.org/XML/1998/namespace}id"
people = {"小野の小町": "ono", "玉造の小町": "tama", "黄泉の使": "messenger", "神将": "guards"}
tei = etree.fromstring(requests.get(url).content)
for span in tei.iter(T + "span"):
    if span.get("type") in ("rt", "rp"):
        span.text = ""
etree.strip_tags(tei, T + "span")
for person in tei.iter(T + "person"):
    person.set(XML_ID, people[person.get(XML_ID)])
for sp in tei.iter(T + "sp"):
    sp.set("who", "#" + people[sp.get("who")[1:]])
speeches = tei.findall(f".//{T}div")[0].findall(T + "sp")
text = {}
for n in (14, 15, 19, 20):
    speaker, *rest = speeches[n - 1]
    text[n] = "".join("".join(e.itertext()) for e in rest)
    print(n, speeches[n - 1].get("who"), speaker.text, text[n])

## 2. 判断を表にする

A〜Dの表現ごとに、`label`（固有名／参照表現／対象外／保留）、`referent`（`ono`／`tama`／`messenger`／`対象外`／`保留`）、`reason`（根拠）を記入してください。相手の判断を見る前に、次の初期方針に従い、各自で判断してください。

- 人物を指す表現を、固有名と参照表現に分ける。名前の条件や一般的な呼称だけなら対象外にする。
- 助詞を除いた表現の範囲と、文脈上の参照先を別々に決める。
- 調べても根拠が足りなければ「判断できない」とする。表には保存値の`保留`を記し、根拠を発言番号とともに書く。

In [ ]:
import pandas as pd

mine = pd.DataFrame([
    ["A", 14, "あなた", "", "", ""],
    ["B", 14, "小町", "", "", ""],
    ["C", 15, "玉造の小町", "", "", ""],
    ["D", 15, "あの人", "", "", ""],
], columns=["case", "speech", "span", "label", "referent", "reason"])
mine

## 3. 相手の判断と比べる

`other`には、比較の例として教材で用意した判断例を入れてあります。一人で進める場合は、この案と比べてください。相手と比べるときは、相手の表の値に書き換えてください。次のセルは、分類か参照先が食い違う行だけを表示します。

In [ ]:
other = pd.DataFrame([
    ["A", "参照表現", "ono"],
    ["B", "対象外", "対象外"],
    ["C", "固有名", "tama"],
    ["D", "参照表現", "tama"],
], columns=["case", "label", "referent"])
both = mine.merge(other, on="case", suffixes=("", "_other"))
both[(both.label != both.label_other) | (both.referent != both.referent_other)]

食い違った行では、範囲・分類・参照先のどれが原因でしょうか。初期方針の規則を一つ具体化し、発言20の「その人」に当てはめてください。判断できない場合は、表に`保留`と記し、不足する根拠を残してください。

改訂した規則に当てはまる例と、当てはまらない例も考えてください。二人称を数える規則にする場合は、たとえば「あなたに頼みます」のような呼びかけの「あなた」をどう扱うか説明してください。

## 4. 判断をTEIのタグで表す

`annotated`は、発言15の二つ目の段落です。固有名なら`persName`、参照表現なら`rs type="person"`を挿入し、`ref`に`#tama`のように、`#`と参照先の人物IDを書いてください。本文の文字は変えないでください。

次のセルは、本文の文字が元と一致する場合に、この段落を戯曲全体のTEIの中で置き換え、加えたタグを表示します。

In [ ]:
annotated = '<persName ref="#tama">玉造の小町</persName>と云う人がいます。あの人を代りにつれて行って下さい。'

target = speeches[14].findall(T + "p")[1]
new = etree.fromstring(f'<p xmlns="http://www.tei-c.org/ns/1.0">{annotated}</p>')
same = "".join(new.itertext()) == "".join(target.itertext())
print("元の本文と一致：", same)
if same:
    speeches[14].replace(target, new)
for e in new:
    print(e.tag.replace(T, ""), e.get("type"), e.get("ref"), "".join(e.itertext()))

## 5. スキーマと参照を確かめる

タグを加えたTEIを、二つの方法で確かめます。

- TEIスキーマ（tei_all、P5 4.12.0）：要素と属性の名前、置ける位置を確かめます。
- 参照の確認：`ref`の人物IDが人物一覧にあるかを確かめます。スキーマは、この対応を確かめません。

4節の`annotated`を変えて、4節から再実行してください。`persName`を`persname`にすると、どちらの確認が変わるでしょうか。`ref`を`#tamatsukuri`にした場合も、実行前に予想してください。

In [ ]:
#@title 操作：スキーマと参照を確かめる
schema = etree.RelaxNG(etree.fromstring(requests.get("https://www.tei-c.org/Vault/P5/4.12.0/xml/tei/custom/schema/relaxng/tei_all.rng").content))
print("TEIスキーマに適合：", schema.validate(tei))
for error in schema.error_log:
    print("  ", error.message)
ids = {person.get(XML_ID) for person in tei.iter(T + "person")}
print("人物一覧のID：", sorted(ids))
for e in tei.iter(T + "persName", T + "rs"):
    if e.get("ref") is not None:
        print(e.get("ref"), "".join(e.itertext()), "人物一覧にある" if e.get("ref")[1:] in ids else "人物一覧にない")

## 確認問題

第2場の発言「やっぱりさっき小野の小町が、……」の「小野の小町」に、固有名の人物タグを付けてください。小野の人物IDは`ono`です。`answer`の文字列にタグを挿入してください。本文の文字は変えないでください。

In [ ]:
answer = 'やっぱりさっき小野の小町が、……'

In [ ]:
#@title 答えを確かめる
import xml.etree.ElementTree as ET

try:
    p = ET.fromstring(f'<p xmlns="http://www.tei-c.org/ns/1.0">{answer}</p>')
except ET.ParseError:
    print("結果：不合格（XMLとして読めません。開始タグと終了タグの対応を確かめてください）")
else:
    tagged = [e for e in p.iter() if e is not p and "小野の小町" in "".join(e.itertext())]
    if "".join(p.itertext()) != "やっぱりさっき小野の小町が、……":
        print("結果：不合格（本文の文字が変わっています。タグだけを加えてください）")
    elif not tagged:
        print("結果：不合格（「小野の小町」にタグがありません）")
    elif tagged[0].tag != "{http://www.tei-c.org/ns/1.0}persName":
        print("結果：不合格（固有名には`persName`を使います）")
    elif "".join(tagged[0].itertext()) != "小野の小町":
        print("結果：不合格（タグの範囲は「小野の小町」だけにしてください）")
    elif tagged[0].get("ref") == "ono":
        print("結果：不合格（`ref`の人物IDの前に`#`を付けてください）")
    elif tagged[0].get("ref") != "#ono" and tagged[0].get("key") != "ono":
        print('結果：不合格（`ref="#ono"`で小野の小町を指してください）')
    else:
        print("結果：合格")